In [ ]:
# FP16：先运行本单元格，再运行下面的评测单元格
import os
os.environ['USE_TF']='0'
os.environ['USE_FLAX']='0'
os.environ['TOKENIZERS_PARALLELISM']='false'
GPU = 1
import gc, re, ast, warnings
from pathlib import Path
import torch
from datasets import load_from_disk
from transformers import AutoTokenizer, AutoModelForCausalLM, set_seed
from transformers.utils import logging as hf_logging
hf_logging.set_verbosity_error()
hf_logging.disable_progress_bar()
warnings.filterwarnings('ignore', category=UserWarning)
set_seed(42)
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False
ROOT = Path('/tf/wjy')
MODEL_PATH = ROOT / 'models/Meta-Llama-3-8B-Instruct'
assert torch.cuda.is_available(), 'CUDA is required'
import time
while torch.cuda.mem_get_info(GPU)[0] < 35 * 1024**3:
    time.sleep(10)
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, local_files_only=True, padding_side='left')
tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(MODEL_PATH, local_files_only=True,
    torch_dtype=torch.float16, device_map={'': GPU}, attn_implementation='sdpa')
model.eval()
assert not getattr(model, 'is_quantized', False)
assert {p.dtype for p in model.parameters() if p.is_floating_point()} == {torch.float16}
EOS_IDS = list({tokenizer.eos_token_id, tokenizer.convert_tokens_to_ids('<|eot_id|>')})
@torch.inference_mode()
def generate_batch(messages, max_new_tokens):
    prompts = [tokenizer.apply_chat_template(m, tokenize=False, add_generation_prompt=True) for m in messages]
    inputs = tokenizer(prompts, return_tensors='pt', padding=True, add_special_tokens=False)
    assert inputs.input_ids.shape[1] + max_new_tokens <= model.config.max_position_embeddings
    inputs = {k: v.to(model.device) for k,v in inputs.items()}
    output = model.generate(**inputs, do_sample=False, temperature=None, top_p=None,
        max_new_tokens=max_new_tokens, eos_token_id=EOS_IDS, pad_token_id=tokenizer.pad_token_id)
    return tokenizer.batch_decode(output[:, inputs['input_ids'].shape[1]:], skip_special_tokens=True)
def release_model():
    global model
    del model
    gc.collect()
    torch.cuda.empty_cache()



In [12]:
from pathlib import Path
import json
script = r'''import argparse, ast, json, os
from pathlib import Path
import numpy as np

p=argparse.ArgumentParser(); p.add_argument('--seed',type=int,required=True); p.add_argument('--gpu',type=int,default=0); p.add_argument('--mask-root',type=Path,required=True); p.add_argument('--output-dir',type=Path,required=True); a=p.parse_args()
os.environ.update({'CUDA_VISIBLE_DEVICES':str(a.gpu),'USE_TF':'0','USE_FLAX':'0','TOKENIZERS_PARALLELISM':'false'})
ROOT=Path('/tf/wjy'); nb=json.loads((ROOT/'humaneval.ipynb').read_text()); md=json.loads((a.mask_root/f'seed_{a.seed}/mask_manifest.json').read_text())
r=np.asarray(np.load(a.mask_root/f'seed_{a.seed}/selected_one_bit_ranks.npy',mmap_mode='r'),dtype=np.int64)
assert md['status']=='mask_ready' and md['direction']=='1to0' and md['target_changed_bits']==83751862 and r.size==83751862 and np.all(r[1:]>=r[:-1])
qs=[]
for c in nb['cells']:
 if c.get('cell_type')!='code': continue
 try: t=ast.parse(''.join(c.get('source',[])))
 except SyntaxError: continue
 for n in t.body:
  if isinstance(n,ast.Assign) and any(isinstance(x,ast.Name) and x.id=='quant_code' for x in n.targets) and isinstance(n.value,ast.Constant) and isinstance(n.value.value,str) and 'load_awq_model' in n.value.value and 'run_evaluation' in n.value.value and 'HumanEval' in n.value.value: qs.append(n.value.value)
assert len(qs)==1, len(qs)
t=ast.parse(qs[0]); assert isinstance(t.body[-1],ast.Try); t.body.pop(); ns={'__name__':'__main__'}; exec(compile(t,'<quant_code>','exec'),ns,ns)
print(f'LOAD clean AWQ W4A16 seed={a.seed}',flush=True); ns['load_awq_model'](); model=ns['model']; model.eval()
import torch
mods=sorted([(n,m,getattr(m,'qweight')) for n,m in model.named_modules() if getattr(m,'qweight',None) is not None],key=lambda x:x[0])
assert len(mods)==224 and all(q.dtype==torch.int32 for _,_,q in mods)
c=np.asarray([q.numel()*32 for _,_,q in mods],dtype=np.int64); cum=np.cumsum(c); assert int(cum[-1])==md['total_valid_payload_bits']
changed=0
for i,(name,m,q) in enumerate(mods):
 b=0 if i==0 else int(cum[i-1]); e=int(cum[i]); lo=int(np.searchsorted(r,b)); hi=int(np.searchsorted(r,e))
 if lo==hi: continue
 z=r[lo:hi]-b; wi=z//32; bi=z%32; flat=q.reshape(-1)
 for w in np.unique(wi):
  old=int(flat[int(w)].item())&0xffffffff
  for bit in bi[wi==w]:
   bit=int(bit); assert (old>>bit)&1==1, (name,int(w),bit); old &= ~(1<<bit); changed+=1
  flat[int(w)]=torch.tensor(old,dtype=torch.int32,device=flat.device)
print(f'INJECTION_OK seed={a.seed} changed={changed} actual_ber={changed/md["total_valid_payload_bits"]:.15f}',flush=True); assert changed==md['target_changed_bits']
ret=ns['run_evaluation'](); out=a.output_dir/f'seed_{a.seed}.json'; out.parent.mkdir(parents=True,exist_ok=True); out.write_text(json.dumps({'status':'completed','seed':a.seed,'correct':ret.get('correct') if isinstance(ret,dict) else None,'total':ret.get('total') if isinstance(ret,dict) else None,'accuracy':ret.get('accuracy') if isinstance(ret,dict) else None,'changed':changed,'actual_ber':changed/md['total_valid_payload_bits'],'evaluator_return':ret},ensure_ascii=False,indent=2,default=str)); print('RESULT_JSON',out,flush=True)
'''
Path('/tf/wjy/humaneval_awq_int4_1to0.py').write_text(script,encoding='utf-8')
print('RUNNER_WRITTEN',len(script))

RUNNER_WRITTEN 3141


In [ ]:
# 已有FP16实测：100/164，pass@1 60.98%（历史结果）
import textwrap


def extract_completion(text, row):
    # Select Python structurally, without consulting reference solutions or tests.
    blocks=re.findall(r'```(?:python|py)?[ \t]*\n(.*?)(?:```|\Z)',text,re.S|re.I)
    candidates=blocks or [text]
    best=None
    for block in candidates:
        lines=block.strip('\n').splitlines()
        starts=[0]+[i for i,line in enumerate(lines) if re.match(r'^(?:async\s+def|def|class|from|import)\b|^@',line)]
        for start in sorted(set(starts)):
            for end in range(len(lines),start,-1):
                snippet='\n'.join(lines[start:end])
                try: tree=ast.parse(snippet)
                except (SyntaxError,IndentationError): continue
                if not any(isinstance(n,(ast.FunctionDef,ast.AsyncFunctionDef)) and n.name==row['entry_point'] for n in tree.body): continue
                if best is None or len(snippet)>len(best): best=snippet
                break
    if best is not None:
        tree=ast.parse(best)
        # Retain imports, constants, helpers and decorators; drop example calls/prose.
        allowed=(ast.Import,ast.ImportFrom,ast.FunctionDef,ast.AsyncFunctionDef,ast.ClassDef,ast.Assign,ast.AnnAssign)
        lines=best.splitlines()
        pieces=[]
        for n in tree.body:
            if isinstance(n,allowed):
                begin=min([n.lineno]+[d.lineno for d in getattr(n,'decorator_list',[])])-1
                pieces.append('\n'.join(lines[begin:n.end_lineno]))
        module='\n\n'.join(pieces)
        # The original prompt retains its imports; the complete implementation overrides its stub.
        return '\n    pass\n\n'+module+'\n'
    # A completion may contain only the function body.
    body=candidates[0].strip('\n')
    for candidate in [body,textwrap.indent(textwrap.dedent(body),'    ')]:
        candidate='\n'+candidate+'\n'
        try: ast.parse(row['prompt']+candidate)
        except (SyntaxError,IndentationError): continue
        return candidate
    return '\n'+body+'\n'

from human_eval.execution import check_correctness
from concurrent.futures import ThreadPoolExecutor


def run_humaneval():
    test = load_from_disk(ROOT / 'humaneval')
    assert len(test) == 164 and len(set(test['task_id'])) == 164
    def messages(r):
        return [{'role':'system','content':'You are an expert Python programmer. Complete the function correctly. Return only valid Python code with the complete function implementation, including any needed imports and helper functions. Do not include explanations.'},
                {'role':'user','content':'Complete the following Python function:\n\n' + r['prompt']}]
    correct=0
    for start in range(0,len(test),4):
        rows=[test[i] for i in range(start,min(start+4,len(test)))]
        generated=generate_batch([messages(r) for r in rows],2048)
        for row,text in zip(rows,generated):
            with ThreadPoolExecutor(max_workers=1) as pool:
                result=pool.submit(check_correctness,dict(row), extract_completion(text,row),10.0).result()
            correct += bool(result['passed'])
    return (correct,len(test))


try:
    humaneval_result = run_humaneval()
finally:
    release_model()
print(f'HumanEval | FP16 | {humaneval_result[0]}/{humaneval_result[1]} | pass@1: {humaneval_result[0]/humaneval_result[1]:.2%}')



In [1]:
# INT8：Quanto W8A16，独立运行，不需要先运行FP16
quant_code = r'''
import os, sysconfig
from pathlib import Path
_nvroot = Path(sysconfig.get_paths()['purelib']) / 'nvidia'
os.environ['CPATH'] = os.pathsep.join((str(p) for p in _nvroot.glob('*/include')))
os.environ['LIBRARY_PATH'] = os.pathsep.join((str(p) for p in _nvroot.glob('*/lib')))
os.environ['MAX_JOBS'] = '2'
os.environ['TORCH_CUDA_ARCH_LIST'] = '8.6;8.9'
GPU = 0
import os
os.environ['USE_TF'] = '0'
os.environ['USE_FLAX'] = '0'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
import gc, re, ast, textwrap, warnings
from pathlib import Path
import torch
from datasets import load_from_disk
from transformers import AutoTokenizer, AutoModelForCausalLM, set_seed
from transformers.utils import logging as hf_logging
from optimum.quanto import quantize, freeze, qint8, quantization_map
from optimum.quanto.nn import QModuleMixin
hf_logging.set_verbosity_error()
hf_logging.disable_progress_bar()
warnings.filterwarnings('ignore', category=UserWarning)
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False
ROOT = Path('/tf/wjy')
MODEL_PATH = ROOT / 'models/Meta-Llama-3-8B-Instruct'
assert torch.cuda.is_available(), 'CUDA is required'
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, local_files_only=True, padding_side='left')
tokenizer.pad_token = tokenizer.eos_token
EOS_IDS = list({tokenizer.eos_token_id, tokenizer.convert_tokens_to_ids('<|eot_id|>')})

def release_model():
    global model
    if 'model' in globals():
        del model
    gc.collect()
    torch.cuda.empty_cache()

def load_quantized_model(bits):
    global model
    assert bits == 8
    release_model()
    set_seed(42)
    model = AutoModelForCausalLM.from_pretrained(MODEL_PATH, local_files_only=True, torch_dtype=torch.float16, device_map={'': GPU}, attn_implementation='sdpa').eval()
    expected = [name for name, module in model.named_modules() if name.startswith('model.layers.') and isinstance(module, torch.nn.Linear)]
    assert len(expected) == model.config.num_hidden_layers * 7
    qtype = qint8
    quantize(model, weights=qtype, activations=None, include=expected)
    freeze(model)
    config = quantization_map(model)
    assert set(config) == set(expected)
    assert all((c == {'weights': f'qint{bits}', 'activations': 'none'} for c in config.values()))
    modules = [model.get_submodule(name) for name in expected]
    assert all((m.frozen and m.weight.dtype == torch.float16 for m in modules))
    assert model.model.embed_tokens.weight.dtype == model.lm_head.weight.dtype == torch.float16
    seen = set()

    def check_input(module, args):
        assert args[0].dtype == torch.float16, 'Activation is not FP16'
        seen.add(id(module))
    hooks = [m.register_forward_pre_hook(check_input) for m in modules]
    try:
        sample = tokenizer('Hello', return_tensors='pt').to(model.device)
        with torch.inference_mode():
            logits = model(**sample, use_cache=False).logits
        assert len(seen) == len(modules) and torch.isfinite(logits).all()
    finally:
        for hook in hooks:
            hook.remove()
    gc.collect()
    torch.cuda.empty_cache()

@torch.inference_mode()
def generate_batch(messages, max_new_tokens):
    prompts = [tokenizer.apply_chat_template(m, tokenize=False, add_generation_prompt=True) for m in messages]
    inputs = tokenizer(prompts, return_tensors='pt', padding=True, add_special_tokens=False)
    assert inputs.input_ids.shape[1] + max_new_tokens <= model.config.max_position_embeddings
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    output = model.generate(**inputs, do_sample=False, temperature=None, top_p=None, max_new_tokens=max_new_tokens, eos_token_id=EOS_IDS, pad_token_id=tokenizer.pad_token_id)
    return tokenizer.batch_decode(output[:, inputs['input_ids'].shape[1]:], skip_special_tokens=True)
import textwrap

import textwrap


def extract_completion(text, row):
    # Select Python structurally, without consulting reference solutions or tests.
    blocks=re.findall(r'```(?:python|py)?[ \t]*\n(.*?)(?:```|\Z)',text,re.S|re.I)
    candidates=blocks or [text]
    best=None
    for block in candidates:
        lines=block.strip('\n').splitlines()
        starts=[0]+[i for i,line in enumerate(lines) if re.match(r'^(?:async\s+def|def|class|from|import)\b|^@',line)]
        for start in sorted(set(starts)):
            for end in range(len(lines),start,-1):
                snippet='\n'.join(lines[start:end])
                try: tree=ast.parse(snippet)
                except (SyntaxError,IndentationError): continue
                if not any(isinstance(n,(ast.FunctionDef,ast.AsyncFunctionDef)) and n.name==row['entry_point'] for n in tree.body): continue
                if best is None or len(snippet)>len(best): best=snippet
                break
    if best is not None:
        tree=ast.parse(best)
        # Retain imports, constants, helpers and decorators; drop example calls/prose.
        allowed=(ast.Import,ast.ImportFrom,ast.FunctionDef,ast.AsyncFunctionDef,ast.ClassDef,ast.Assign,ast.AnnAssign)
        lines=best.splitlines()
        pieces=[]
        for n in tree.body:
            if isinstance(n,allowed):
                begin=min([n.lineno]+[d.lineno for d in getattr(n,'decorator_list',[])])-1
                pieces.append('\n'.join(lines[begin:n.end_lineno]))
        module='\n\n'.join(pieces)
        # The original prompt retains its imports; the complete implementation overrides its stub.
        return '\n    pass\n\n'+module+'\n'
    # A completion may contain only the function body.
    body=candidates[0].strip('\n')
    for candidate in [body,textwrap.indent(textwrap.dedent(body),'    ')]:
        candidate='\n'+candidate+'\n'
        try: ast.parse(row['prompt']+candidate)
        except (SyntaxError,IndentationError): continue
        return candidate
    return '\n'+body+'\n'

from human_eval.execution import check_correctness
from concurrent.futures import ThreadPoolExecutor


def run_humaneval():
    test = load_from_disk(ROOT / 'humaneval')
    assert len(test) == 164 and len(set(test['task_id'])) == 164
    def messages(r):
        return [{'role':'system','content':'You are an expert Python programmer. Complete the function correctly. Return only valid Python code with the complete function implementation, including any needed imports and helper functions. Do not include explanations.'},
                {'role':'user','content':'Complete the following Python function:\n\n' + r['prompt']}]
    correct=0
    for start in range(0,len(test),4):
        rows=[test[i] for i in range(start,min(start+4,len(test)))]
        generated=generate_batch([messages(r) for r in rows],2048)
        for row,text in zip(rows,generated):
            with ThreadPoolExecutor(max_workers=1) as pool:
                result=pool.submit(check_correctness,dict(row), extract_completion(text,row),10.0).result()
            correct += bool(result['passed'])
    return (correct,len(test))


run_evaluation = run_humaneval
try:
    load_quantized_model(8)
    correct, total = run_evaluation()
    print(f'HumanEval | Quanto W8A16 | {correct}/{total} | accuracy: {correct / total:.2%}', flush=True)
finally:
    release_model()
'''
import os, subprocess
from pathlib import Path
from collections import deque
_root = Path('/tf/wjy')
_python = _root / '.venv-quant/bin/python'
_env = os.environ.copy()
_env['CUDA_VISIBLE_DEVICES'] = '1'
_env['PATH'] = str(_python.parent) + os.pathsep + _env.get('PATH', '')
_env['PYTHONWARNINGS'] = 'ignore'
for key, folder in [('TMPDIR', '.tmp'), ('HF_HOME', '.cache/huggingface'), ('TORCH_HOME', '.cache/torch'), ('TORCH_EXTENSIONS_DIR', '.cache/torch_extensions')]:
    directory = _root / folder
    directory.mkdir(parents=True, exist_ok=True)
    _env[key] = str(directory)
_tail = deque(maxlen=25)
with subprocess.Popen([str(_python), '-u', '-c', quant_code], cwd=str(_root), env=_env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1) as _p:
    try:
        for line in _p.stdout:
            _tail.append(line)
            if line.startswith(('HumanEval |', 'MathQA |', 'MMLU |')):
                print(line, end='', flush=True)
        if _p.wait() != 0:
            raise RuntimeError('评测失败，末尾诊断：' + ''.join(_tail))
    except BaseException:
        if _p.poll() is None:
            _p.terminate()
            _p.wait()
        raise

HumanEval | Quanto W8A16 | 100/164 | accuracy: 60.98%


In [ ]:
# INT4：AWQ W4A16，独立运行，不需要先运行FP16或INT8
# WikiText-2 train校准128条，不使用HumanEval测试题；只打印最终分数
quant_code = r'''
import os, sysconfig
from pathlib import Path
_nvroot = Path(sysconfig.get_paths()['purelib']) / 'nvidia'
os.environ['CPATH'] = os.pathsep.join((str(p) for p in _nvroot.glob('*/include')))
os.environ['LIBRARY_PATH'] = os.pathsep.join((str(p) for p in _nvroot.glob('*/lib')))
os.environ['MAX_JOBS'] = '2'
os.environ['TORCH_CUDA_ARCH_LIST'] = '8.6;8.9'
GPU = 0
import os
os.environ['USE_TF'] = '0'
os.environ['USE_FLAX'] = '0'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
import gc, re, ast, textwrap, warnings
from pathlib import Path
import torch
from datasets import load_from_disk
from transformers import AutoTokenizer, AutoModelForCausalLM, set_seed
from transformers.utils import logging as hf_logging
hf_logging.set_verbosity_error()
hf_logging.disable_progress_bar()
warnings.filterwarnings('ignore', category=UserWarning)
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False
ROOT = Path('/tf/wjy')
MODEL_PATH = ROOT / 'models/Meta-Llama-3-8B-Instruct'
assert torch.cuda.is_available(), 'CUDA is required'
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, local_files_only=True, padding_side='left')
tokenizer.pad_token = tokenizer.eos_token
EOS_IDS = list({tokenizer.eos_token_id, tokenizer.convert_tokens_to_ids('<|eot_id|>')})

def release_model():
    global model
    if 'model' in globals():
        del model
    gc.collect()
    torch.cuda.empty_cache()

@torch.inference_mode()
def generate_batch(messages, max_new_tokens):
    prompts = [tokenizer.apply_chat_template(m, tokenize=False, add_generation_prompt=True) for m in messages]
    inputs = tokenizer(prompts, return_tensors='pt', padding=True, add_special_tokens=False)
    assert inputs.input_ids.shape[1] + max_new_tokens <= model.config.max_position_embeddings
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    output = model.generate(**inputs, do_sample=False, temperature=None, top_p=None, max_new_tokens=max_new_tokens, eos_token_id=EOS_IDS, pad_token_id=tokenizer.pad_token_id)
    return tokenizer.batch_decode(output[:, inputs['input_ids'].shape[1]:], skip_special_tokens=True)
import textwrap

import textwrap


def extract_completion(text, row):
    # Select Python structurally, without consulting reference solutions or tests.
    blocks=re.findall(r'```(?:python|py)?[ \t]*\n(.*?)(?:```|\Z)',text,re.S|re.I)
    candidates=blocks or [text]
    best=None
    for block in candidates:
        lines=block.strip('\n').splitlines()
        starts=[0]+[i for i,line in enumerate(lines) if re.match(r'^(?:async\s+def|def|class|from|import)\b|^@',line)]
        for start in sorted(set(starts)):
            for end in range(len(lines),start,-1):
                snippet='\n'.join(lines[start:end])
                try: tree=ast.parse(snippet)
                except (SyntaxError,IndentationError): continue
                if not any(isinstance(n,(ast.FunctionDef,ast.AsyncFunctionDef)) and n.name==row['entry_point'] for n in tree.body): continue
                if best is None or len(snippet)>len(best): best=snippet
                break
    if best is not None:
        tree=ast.parse(best)
        # Retain imports, constants, helpers and decorators; drop example calls/prose.
        allowed=(ast.Import,ast.ImportFrom,ast.FunctionDef,ast.AsyncFunctionDef,ast.ClassDef,ast.Assign,ast.AnnAssign)
        lines=best.splitlines()
        pieces=[]
        for n in tree.body:
            if isinstance(n,allowed):
                begin=min([n.lineno]+[d.lineno for d in getattr(n,'decorator_list',[])])-1
                pieces.append('\n'.join(lines[begin:n.end_lineno]))
        module='\n\n'.join(pieces)
        # The original prompt retains its imports; the complete implementation overrides its stub.
        return '\n    pass\n\n'+module+'\n'
    # A completion may contain only the function body.
    body=candidates[0].strip('\n')
    for candidate in [body,textwrap.indent(textwrap.dedent(body),'    ')]:
        candidate='\n'+candidate+'\n'
        try: ast.parse(row['prompt']+candidate)
        except (SyntaxError,IndentationError): continue
        return candidate
    return '\n'+body+'\n'

from human_eval.execution import check_correctness
from concurrent.futures import ThreadPoolExecutor


def run_humaneval():
    test = load_from_disk(ROOT / 'humaneval')
    assert len(test) == 164 and len(set(test['task_id'])) == 164
    def messages(r):
        return [{'role':'system','content':'You are an expert Python programmer. Complete the function correctly. Return only valid Python code with the complete function implementation, including any needed imports and helper functions. Do not include explanations.'},
                {'role':'user','content':'Complete the following Python function:\n\n' + r['prompt']}]
    correct=0
    for start in range(0,len(test),4):
        rows=[test[i] for i in range(start,min(start+4,len(test)))]
        generated=generate_batch([messages(r) for r in rows],2048)
        for row,text in zip(rows,generated):
            with ThreadPoolExecutor(max_workers=1) as pool:
                result=pool.submit(check_correctness,dict(row), extract_completion(text,row),10.0).result()
            correct += bool(result['passed'])
    return (correct,len(test))


run_evaluation = run_humaneval

def load_awq_model():
    global model
    from datasets import load_dataset
    from awq import AutoAWQForCausalLM
    from awq.modules.linear import WQLinear_GEMM
    set_seed(42)
    raw = load_dataset('Salesforce/wikitext', 'wikitext-2-raw-v1', split='train', cache_dir=str(ROOT / '.cache/huggingface/datasets'))
    calibration = []
    seen_text = set()
    for row in raw:
        text = row['text'].strip()
        ids = tokenizer.encode(text, add_special_tokens=False)
        if 32 <= len(ids) <= 512 and text not in seen_text:
            calibration.append(text)
            seen_text.add(text)
        if len(calibration) == 128:
            break
    assert len(calibration) == 128
    awq_model = AutoAWQForCausalLM.from_pretrained(str(MODEL_PATH), torch_dtype=torch.float16, trust_remote_code=False, local_files_only=True, device_map={'': 'cpu'}, attn_implementation='sdpa')
    awq_model.quantize(tokenizer, quant_config={'zero_point': True, 'q_group_size': 128, 'w_bit': 4, 'version': 'GEMM'}, calib_data=calibration, max_calib_samples=128, max_calib_seq_len=512, n_parallel_calib_samples=4, max_chunk_memory=268435456, export_compatible=False)
    model = awq_model.model.to('cuda:0').eval()
    if hasattr(awq_model, 'quantizer'):
        del awq_model.quantizer
    layers = [m for m in model.modules() if isinstance(m, WQLinear_GEMM)]
    assert len(layers) == model.config.num_hidden_layers * 7
    assert all((m.w_bit == 4 and m.qweight.dtype == torch.int32 and (m.scales.dtype == torch.float16) for m in layers))
    seen = set()

    def verify(m, args):
        assert args[0].dtype == torch.float16
        seen.add(id(m))
    hooks = [m.register_forward_pre_hook(verify) for m in layers]
    try:
        with torch.inference_mode():
            logits = model(**tokenizer('Hello', return_tensors='pt').to('cuda:0'), use_cache=False).logits
        assert len(seen) == len(layers) and torch.isfinite(logits).all()
    finally:
        for hook in hooks:
            hook.remove()
    gc.collect()
    torch.cuda.empty_cache()
try:
    load_awq_model()
    correct, total = run_evaluation()
    print(f'HumanEval | AWQ W4A16 | {correct}/{total} | accuracy: {correct / total:.2%}', flush=True)
finally:
    release_model()
'''
import os, subprocess
from pathlib import Path
from collections import deque
_root = Path('/tf/wjy')
_python = _root / '.venv-quant/bin/python'
_env = os.environ.copy()
_env['CUDA_VISIBLE_DEVICES'] = '1'
_env['PATH'] = str(_python.parent) + os.pathsep + _env.get('PATH', '')
_env['PYTHONWARNINGS'] = 'ignore'
for key, folder in [('TMPDIR', '.tmp'), ('HF_HOME', '.cache/huggingface'), ('TORCH_HOME', '.cache/torch'), ('TORCH_EXTENSIONS_DIR', '.cache/torch_extensions')]:
    directory = _root / folder
    directory.mkdir(parents=True, exist_ok=True)
    _env[key] = str(directory)
_tail = deque(maxlen=25)
with subprocess.Popen([str(_python), '-u', '-c', quant_code], cwd=str(_root), env=_env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1) as _p:
    try:
        for line in _p.stdout:
            _tail.append(line)
            if line.startswith(('HumanEval |', 'MathQA |', 'MMLU |')):
                print(line, end='', flush=True)
        if _p.wait() != 0:
            raise RuntimeError('评测失败，末尾诊断：' + ''.join(_tail))
    except BaseException:
        if _p.poll() is None:
            _p.terminate()
            _p.wait()
        raise

In [7]:
# Paste this entire file into ONE NEW cell at the end of /tf/wjy/humaneval.ipynb.
# Existing cells are read only. Only INT8 weight payloads receive faults.
import ast as _fi_ast
import json as _fi_json
import os as _fi_os
import subprocess as _fi_sp
from pathlib import Path as _fi_Path
from collections import deque as _fi_deque

_fi_root = _fi_Path('/tf/wjy')
_fi_python = _fi_root / '.venv-quant/bin/python'
_fi_gpu = '0'  # Physical GPU; check available memory before running.
_fi_seeds = (2026091201, 2026091202, 2026091203, 2026091204, 2026091205)
assert _fi_python.is_file(), f'Python not found: {_fi_python}'

# Reuse the saved, original Quanto loader and HumanEval evaluator without executing
# the original cell's subprocess wrapper or changing its source.
_fi_nb = _fi_json.loads((_fi_root / 'humaneval.ipynb').read_text(encoding='utf-8'))
_fi_candidates = []
for _fi_cell in _fi_nb['cells']:
    _fi_source = ''.join(_fi_cell.get('source', []))
    if _fi_cell.get('cell_type') != 'code':
        continue
    try:
        _fi_outer = _fi_ast.parse(_fi_source)
    except SyntaxError:
        continue
    for _fi_node in _fi_outer.body:
        if (isinstance(_fi_node, _fi_ast.Assign)
                and any(isinstance(t, _fi_ast.Name) and t.id == 'quant_code'
                        for t in _fi_node.targets)
                and isinstance(_fi_node.value, _fi_ast.Constant)
                and isinstance(_fi_node.value.value, str)):
            _fi_inner = _fi_ast.parse(_fi_node.value.value)
            _fi_names = {n.name for n in _fi_inner.body
                         if isinstance(n, _fi_ast.FunctionDef)}
            if 'load_quantized_model' in _fi_names and 'HumanEval |' in _fi_node.value.value:
                _fi_candidates.append(_fi_inner)
assert len(_fi_candidates) == 1, 'Expected exactly one original Quanto INT8 cell; save the notebook and check its code.'
_fi_tree = _fi_candidates[0]
assert isinstance(_fi_tree.body[-1], _fi_ast.Try), 'Unexpected original INT8 cell structure.'
assert any(isinstance(n, _fi_ast.Call) and isinstance(n.func, _fi_ast.Name)
           and n.func.id == 'load_quantized_model'
           for n in _fi_ast.walk(_fi_tree.body[-1])), 'Original INT8 entry point not found.'
_fi_entry = _fi_tree.body.pop()
_fi_load_calls = [n for n in _fi_ast.walk(_fi_entry)
                  if isinstance(n, _fi_ast.Call) and isinstance(n.func, _fi_ast.Name)
                  and n.func.id == 'load_quantized_model']
assert len(_fi_load_calls) == 1, 'Expected a separate INT8 entry point.'
assert len(_fi_load_calls[0].args) == 1 and _fi_ast.literal_eval(_fi_load_calls[0].args[0]) == 8, 'Expected INT8 only.'
_fi_entry_source = _fi_ast.unparse(_fi_entry)
_fi_base = _fi_ast.unparse(_fi_tree)

_fi_extra = r'''
import numpy as np
import sys

def sample_global_bits(total, count, seed):
    # Uniform draws over ALL matrices' bit addresses. Reject repeats until
    # exactly count unique addresses remain; no fixed per-matrix quota.
    rng = np.random.default_rng(seed)
    chosen = np.unique(rng.integers(0, total, size=count, dtype=np.int64))
    while chosen.size < count:
        extra = rng.integers(0, total, size=count - chosen.size, dtype=np.int64)
        chosen = np.union1d(chosen, extra)
    assert chosen.size == count
    return chosen

def flip_int8_weights(seed):
    config = quantization_map(model)
    assert config and all(c == {'weights': 'qint8', 'activations': 'none'}
                          for c in config.values())
    payloads = []
    storage_keys = set()
    for name in sorted(config):
        module = model.get_submodule(name)
        weight = module.weight
        assert module.frozen and weight.qtype == qint8, name
        raw = weight._data
        assert raw.dtype == torch.int8 and raw.numel() == weight.numel(), name
        assert raw.ndim == 2, name
        key = (str(raw.device), raw.untyped_storage().data_ptr())
        assert key not in storage_keys, 'Shared INT8 storage must not be counted twice.'
        storage_keys.add(key)
        payloads.append((name, raw))
    assert len(payloads) == model.config.num_hidden_layers * 7
    total = sum(raw.numel() * 8 for _, raw in payloads)
    # Integer arithmetic: nearest integer to total * 0.003.
    count = (total * 3 + 500) // 1000
    chosen = sample_global_bits(total, count, seed)
    popcount = np.array([i.bit_count() for i in range(256)], dtype=np.uint8)
    offset = 0
    verified = 0
    with torch.no_grad():
        for name, raw in payloads:
            end = offset + raw.numel() * 8
            lo, hi = np.searchsorted(chosen, [offset, end])
            local = chosen[lo:hi] - offset
            # CPU copy explicitly copied back, including noncontiguous payloads.
            before = raw.detach().cpu().contiguous().numpy().view(np.uint8).reshape(-1).copy()
            mask = np.zeros(before.size, dtype=np.uint8)
            bit_masks = np.left_shift(np.uint8(1), (local % 8).astype(np.uint8))
            # at() handles multiple DIFFERENT bits in the same weight byte.
            np.bitwise_xor.at(mask, local // 8, bit_masks)
            assert int(popcount[mask].sum(dtype=np.int64)) == hi - lo
            after = np.bitwise_xor(before, mask)
            raw.copy_(torch.from_numpy(after.view(np.int8).reshape(tuple(raw.shape))).to(raw.device))
            actual = raw.detach().cpu().contiguous().numpy().view(np.uint8).reshape(-1)
            assert np.array_equal(actual, after), f'Write-back failed: {name}'
            verified += int(popcount[np.bitwise_xor(before, actual)].sum(dtype=np.int64))
            offset = end
            del local, before, mask, bit_masks, after, actual
    assert offset == total and verified == count
    del chosen, payloads
    gc.collect()
    torch.cuda.empty_cache()
    return count, total

def check_finite_output(module, args, output):
    # Do not report an argmax of NaNs as a valid evaluation score.
    if not torch.isfinite(output.logits[:, -1, :]).all().item():
        raise RuntimeError('Nonfinite logits after fault injection; accuracy is invalid.')

from optimum.quanto import quantization_map, qint8
import gc
trial = int(sys.argv[1])
seed = int(sys.argv[2])
original_load_quantized_model = load_quantized_model

def load_quantized_model(bits):
    assert bits == 8
    original_load_quantized_model(bits)
    flipped, total_bits = flip_int8_weights(seed)
    model.register_forward_hook(check_finite_output)
    print(f'FI_BITS|{flipped}/{total_bits}', flush=True)
'''
_fi_program = _fi_base + '\n' + _fi_extra + '\n' + _fi_entry_source
compile(_fi_program, '<HumanEval_INT8_bitflip>', 'exec')

_fi_env = _fi_os.environ.copy()
_fi_env.update(CUDA_VISIBLE_DEVICES=_fi_gpu, PYTHONWARNINGS='ignore',
               USE_TF='0', USE_FLAX='0', TOKENIZERS_PARALLELISM='false')
_fi_env['PATH'] = str(_fi_python.parent) + _fi_os.pathsep + _fi_env.get('PATH', '')
for _fi_key, _fi_folder in [
    ('TMPDIR', '.tmp'), ('HF_HOME', '.cache/huggingface'),
    ('TORCH_HOME', '.cache/torch'), ('TORCH_EXTENSIONS_DIR', '.cache/torch_extensions')
]:
    _fi_dir = _fi_root / _fi_folder
    _fi_dir.mkdir(parents=True, exist_ok=True)
    _fi_env[_fi_key] = str(_fi_dir)

# Independent sequential processes: no accumulated faults or retained GPU model.
# No model checkpoints or separate result files are written.
for _fi_trial, _fi_seed in enumerate(_fi_seeds, 1):
    _fi_tail = _fi_deque(maxlen=30)
    _fi_result = []
    _fi_bits = 'unknown'
    with _fi_sp.Popen(
        [str(_fi_python), '-u', '-c', _fi_program, str(_fi_trial), str(_fi_seed)],
        cwd=str(_fi_root), env=_fi_env, stdout=_fi_sp.PIPE,
        stderr=_fi_sp.STDOUT, text=True, bufsize=1,
    ) as _fi_process:
        try:
            for _fi_line in _fi_process.stdout:
                _fi_tail.append(_fi_line.rstrip())
                if _fi_line.startswith('FI_BITS|'):
                    _fi_bits = _fi_line.strip().split('|', 1)[1]
                if _fi_line.startswith('HumanEval |'):
                    _fi_result.append(_fi_line.rstrip())
            _fi_rc = _fi_process.wait()
        except BaseException:
            if _fi_process.poll() is None:
                _fi_process.terminate()
                try:
                    _fi_process.wait(timeout=10)
                except _fi_sp.TimeoutExpired:
                    _fi_process.kill()
                    _fi_process.wait()
            raise
    if _fi_rc == 0 and len(_fi_result) == 1:
        print(f'HumanEval | INT8 bit flip | round {_fi_trial}/5 | seed={_fi_seed} | '
              f'flips={_fi_bits} | ' + _fi_result[0].split('|', 1)[1].strip(), flush=True)
    else:
        _fi_reason = next((s for s in reversed(_fi_tail) if s.strip()), 'No diagnostic output')
        print(f'HumanEval | INT8 bit flip | round {_fi_trial}/5 | seed={_fi_seed} | '
              f'FAILED: {_fi_reason[:500]}', flush=True)
        break  # Stop on infrastructure errors; do not waste four more loads.


HumanEval | INT8 bit flip | round 1/5 | seed=2026091201 | flips=167503725/55834574848 | Quanto W8A16 | 0/164 | accuracy: 0.00%
HumanEval | INT8 bit flip | round 2/5 | seed=2026091202 | flips=167503725/55834574848 | Quanto W8A16 | 0/164 | accuracy: 0.00%
HumanEval | INT8 bit flip | round 3/5 | seed=2026091203 | flips=167503725/55834574848 | Quanto W8A16 | 0/164 | accuracy: 0.00%


KeyboardInterrupt: 

In [8]:
# Paste this entire file into ONE NEW cell at the end of /tf/wjy/humaneval.ipynb.
# Existing cells are read only. Only INT8 weight payloads receive faults.
import ast as _fi_ast
import json as _fi_json
import os as _fi_os
import subprocess as _fi_sp
from pathlib import Path as _fi_Path
from collections import deque as _fi_deque

_fi_root = _fi_Path('/tf/wjy')
_fi_python = _fi_root / '.venv-quant/bin/python'
_fi_gpu = '0'  # Physical GPU; check available memory before running.
_fi_seeds = (2026091201, 2026091202, 2026091203, 2026091204, 2026091205)
assert _fi_python.is_file(), f'Python not found: {_fi_python}'

# Reuse the saved, original Quanto loader and HumanEval evaluator without executing
# the original cell's subprocess wrapper or changing its source.
_fi_nb = _fi_json.loads((_fi_root / 'humaneval.ipynb').read_text(encoding='utf-8'))
_fi_candidates = []
for _fi_cell in _fi_nb['cells']:
    _fi_source = ''.join(_fi_cell.get('source', []))
    if _fi_cell.get('cell_type') != 'code':
        continue
    try:
        _fi_outer = _fi_ast.parse(_fi_source)
    except SyntaxError:
        continue
    for _fi_node in _fi_outer.body:
        if (isinstance(_fi_node, _fi_ast.Assign)
                and any(isinstance(t, _fi_ast.Name) and t.id == 'quant_code'
                        for t in _fi_node.targets)
                and isinstance(_fi_node.value, _fi_ast.Constant)
                and isinstance(_fi_node.value.value, str)):
            _fi_inner = _fi_ast.parse(_fi_node.value.value)
            _fi_names = {n.name for n in _fi_inner.body
                         if isinstance(n, _fi_ast.FunctionDef)}
            if 'load_quantized_model' in _fi_names and 'HumanEval |' in _fi_node.value.value:
                _fi_candidates.append(_fi_inner)
assert len(_fi_candidates) == 1, 'Expected exactly one original Quanto INT8 cell; save the notebook and check its code.'
_fi_tree = _fi_candidates[0]
assert isinstance(_fi_tree.body[-1], _fi_ast.Try), 'Unexpected original INT8 cell structure.'
assert any(isinstance(n, _fi_ast.Call) and isinstance(n.func, _fi_ast.Name)
           and n.func.id == 'load_quantized_model'
           for n in _fi_ast.walk(_fi_tree.body[-1])), 'Original INT8 entry point not found.'
_fi_entry = _fi_tree.body.pop()
_fi_load_calls = [n for n in _fi_ast.walk(_fi_entry)
                  if isinstance(n, _fi_ast.Call) and isinstance(n.func, _fi_ast.Name)
                  and n.func.id == 'load_quantized_model']
assert len(_fi_load_calls) == 1, 'Expected a separate INT8 entry point.'
assert len(_fi_load_calls[0].args) == 1 and _fi_ast.literal_eval(_fi_load_calls[0].args[0]) == 8, 'Expected INT8 only.'
_fi_entry_source = _fi_ast.unparse(_fi_entry)
_fi_base = _fi_ast.unparse(_fi_tree)

# Instrument only an in-memory copy of the original evaluator.
_diag_found = 0
for _diag_fn in _fi_tree.body:
    if isinstance(_diag_fn, _fi_ast.FunctionDef) and _diag_fn.name == 'run_humaneval':
        for _diag_i, _diag_node in enumerate(_diag_fn.body):
            if (isinstance(_diag_node, _fi_ast.Assign)
                    and any(isinstance(t, _fi_ast.Name) and t.id == 'correct' for t in _diag_node.targets)):
                _diag_insert = _fi_ast.parse(r"""
test = test.select(range(min(3, len(test)))) if hasattr(test, 'select') else test[:3]
print('DIAG | task_ids=' + repr([r['task_id'] for r in test]), flush=True)
fixture = {'task_id': 'diagnostic/add', 'prompt': 'def add(a, b):\n',
           'entry_point': 'add', 'test': 'def check(candidate):\n    assert candidate(2, 3) == 5\n    assert candidate(-1, 1) == 0\n'}
with ThreadPoolExecutor(max_workers=1) as probe_pool:
    probe = probe_pool.submit(check_correctness, fixture, '    return a + b\n', 10.0).result()
print('DIAG | known_correct_checker=' + repr(probe), flush=True)
assert probe['passed'], 'Checker failed on known correct code; do not interpret model scores.'
""").body
                _diag_fn.body[_diag_i:_diag_i] = _diag_insert
                _diag_found += 1
                break
assert _diag_found == 1, 'Evaluator structure differs; send the original run_humaneval function.'

class _DiagResults(_fi_ast.NodeTransformer):
    def visit_Assign(self, node):
        self.generic_visit(node)
        if any(isinstance(t, _fi_ast.Name) and t.id == 'result' for t in node.targets):
            return [node, _fi_ast.parse("print('DIAG | checker_result=' + repr(result), flush=True)").body[0]]
        return node

_fi_tree = _DiagResults().visit(_fi_tree)
_fi_ast.fix_missing_locations(_fi_tree)
_fi_base = _fi_ast.unparse(_fi_tree)
_fi_extra = r'''

import numpy as np
import sys

def sample_global_bits(total, count, seed):
    # Uniform draws over ALL matrices' bit addresses. Reject repeats until
    # exactly count unique addresses remain; no fixed per-matrix quota.
    rng = np.random.default_rng(seed)
    chosen = np.unique(rng.integers(0, total, size=count, dtype=np.int64))
    while chosen.size < count:
        extra = rng.integers(0, total, size=count - chosen.size, dtype=np.int64)
        chosen = np.union1d(chosen, extra)
    assert chosen.size == count
    return chosen

def flip_int8_weights(seed):
    config = quantization_map(model)
    assert config and all(c == {'weights': 'qint8', 'activations': 'none'}
                          for c in config.values())
    payloads = []
    storage_keys = set()
    for name in sorted(config):
        module = model.get_submodule(name)
        weight = module.weight
        assert module.frozen and weight.qtype == qint8, name
        raw = weight._data
        assert raw.dtype == torch.int8 and raw.numel() == weight.numel(), name
        assert raw.ndim == 2, name
        key = (str(raw.device), raw.untyped_storage().data_ptr())
        assert key not in storage_keys, 'Shared INT8 storage must not be counted twice.'
        storage_keys.add(key)
        payloads.append((name, raw))
    assert len(payloads) == model.config.num_hidden_layers * 7
    total = sum(raw.numel() * 8 for _, raw in payloads)
    # Integer arithmetic: nearest integer to total * 0.003.
    count = (total * 3 + 500) // 1000
    chosen = sample_global_bits(total, count, seed)
    popcount = np.array([i.bit_count() for i in range(256)], dtype=np.uint8)
    offset = 0
    verified = 0
    with torch.no_grad():
        for name, raw in payloads:
            end = offset + raw.numel() * 8
            lo, hi = np.searchsorted(chosen, [offset, end])
            local = chosen[lo:hi] - offset
            # CPU copy explicitly copied back, including noncontiguous payloads.
            before = raw.detach().cpu().contiguous().numpy().view(np.uint8).reshape(-1).copy()
            mask = np.zeros(before.size, dtype=np.uint8)
            bit_masks = np.left_shift(np.uint8(1), (local % 8).astype(np.uint8))
            # at() handles multiple DIFFERENT bits in the same weight byte.
            np.bitwise_xor.at(mask, local // 8, bit_masks)
            assert int(popcount[mask].sum(dtype=np.int64)) == hi - lo
            after = np.bitwise_xor(before, mask)
            raw.copy_(torch.from_numpy(after.view(np.int8).reshape(tuple(raw.shape))).to(raw.device))
            actual = raw.detach().cpu().contiguous().numpy().view(np.uint8).reshape(-1)
            assert np.array_equal(actual, after), f'Write-back failed: {name}'
            verified += int(popcount[np.bitwise_xor(before, actual)].sum(dtype=np.int64))
            offset = end
            del local, before, mask, bit_masks, after, actual
    assert offset == total and verified == count
    del chosen, payloads
    gc.collect()
    torch.cuda.empty_cache()
    return count, total

def check_finite_output(module, args, output):
    # Do not report an argmax of NaNs as a valid evaluation score.
    if not torch.isfinite(output.logits[:, -1, :]).all().item():
        raise RuntimeError('Nonfinite logits after fault injection; accuracy is invalid.')

from optimum.quanto import quantization_map, qint8
import gc

import time
mode = sys.argv[1]
print(f'DIAG | mode={mode} | loading clean INT8', flush=True)
load_quantized_model(8)
print('DIAG | model loaded', flush=True)
try:
    if mode == 'fault':
        print('DIAG | sampling and flipping bits', flush=True)
        k, n = flip_int8_weights(2026091201)
        print(f'DIAG | verified flips={k}/{n}', flush=True)
    hook = model.register_forward_hook(check_finite_output)
    original_generate = generate_batch
    original_extract = extract_completion

    def generate_batch(messages, max_new_tokens):
        print(f'DIAG | generating {len(messages)} tasks, max_new_tokens={max_new_tokens}', flush=True)
        started = time.monotonic()
        output = original_generate(messages, max_new_tokens)
        print(f'DIAG | generation finished in {time.monotonic()-started:.1f}s', flush=True)
        for i, text in enumerate(output):
            print(f'DIAG | raw[{i}] | chars={len(text)} | preview={text[:1400]!r}', flush=True)
        return output

    def extract_completion(text, row):
        value = original_extract(text, row)
        print(f"DIAG | extracted | {row['task_id']} | chars={len(value)} | preview={value[:1400]!r}", flush=True)
        return value

    correct, total = run_humaneval()
    print(f'DIAG | mode={mode} | {correct}/{total}', flush=True)
finally:
    release_model()

'''
_fi_program = _fi_base + '\n' + _fi_extra
compile(_fi_program, '<HumanEval_diagnostic>', 'exec')
_fi_env = _fi_os.environ.copy()
_fi_env.update(CUDA_VISIBLE_DEVICES=_fi_gpu, PYTHONWARNINGS='ignore',
               USE_TF='0', USE_FLAX='0', TOKENIZERS_PARALLELISM='false')
_fi_env['PATH'] = str(_fi_python.parent) + _fi_os.pathsep + _fi_env.get('PATH', '')
for _fi_key, _fi_folder in [
    ('TMPDIR', '.tmp'), ('HF_HOME', '.cache/huggingface'),
    ('TORCH_HOME', '.cache/torch'), ('TORCH_EXTENSIONS_DIR', '.cache/torch_extensions')
]:
    _fi_dir = _fi_root / _fi_folder
    _fi_dir.mkdir(parents=True, exist_ok=True)
    _fi_env[_fi_key] = str(_fi_dir)


# Run only after interrupting the previous five-round cell or letting it finish.
for _diag_mode in ('clean', 'fault'):
    print(f'DIAG | starting {_diag_mode}', flush=True)
    _fi_tail = _fi_deque(maxlen=30)
    with _fi_sp.Popen([str(_fi_python), '-u', '-c', _fi_program, _diag_mode],
                      cwd=str(_fi_root), env=_fi_env, stdout=_fi_sp.PIPE,
                      stderr=_fi_sp.STDOUT, text=True, bufsize=1) as _fi_process:
        try:
            for _fi_line in _fi_process.stdout:
                _fi_tail.append(_fi_line.rstrip())
                if _fi_line.startswith('DIAG |'):
                    print(_fi_line, end='', flush=True)
            if _fi_process.wait() != 0:
                raise RuntimeError('Diagnostic failed:\n' + '\n'.join(_fi_tail))
        except BaseException:
            if _fi_process.poll() is None:
                _fi_process.terminate()
                try:
                    _fi_process.wait(timeout=10)
                except _fi_sp.TimeoutExpired:
                    _fi_process.kill()
                    _fi_process.wait()
            raise


DIAG | starting clean
DIAG | mode=clean | loading clean INT8
DIAG | model loaded
DIAG | task_ids=['HumanEval/0', 'HumanEval/1', 'HumanEval/2']
DIAG | known_correct_checker={'task_id': 'diagnostic/add', 'passed': True, 'result': 'passed', 'completion_id': None}
DIAG | generating 3 tasks, max_new_tokens=2048
DIAG | generation finished in 8.9s
DIAG | raw[0] | chars=285 | preview='```\nfrom typing import List\n\ndef has_close_elements(numbers: List[float], threshold: float) -> bool:\n    for i in range(len(numbers)):\n        for j in range(i + 1, len(numbers)):\n            if abs(numbers[i] - numbers[j]) < threshold:\n                return True\n    return False\n```'
DIAG | raw[1] | chars=644 | preview="```\nfrom typing import List\n\ndef separate_paren_groups(paren_string: str) -> List[str]:\n    stack = []\n    groups = []\n    current_group = ''\n    \n    for char in paren_string:\n        if char =='':\n            continue\n        if char == '(':\n            stack.append(char)\

In [9]:
import os
import subprocess
from pathlib import Path

ROOT = Path("/tf/wjy")
PYTHON = ROOT / ".venv-quant/bin/python"
MASK_ROOT = ROOT / "awq_int4_1to0_ber003_masks_v2_t5n573zd"
OUT = ROOT / "results_awq_int4_humaneval_1to0"
RUNNER = ROOT / "humaneval_awq_int4_1to0.py"

assert PYTHON.is_file()
assert MASK_ROOT.is_dir()
assert RUNNER.is_file(), f"请先保存 HumanEval 独立运行脚本：{RUNNER}"

OUT.mkdir(parents=True, exist_ok=True)

env = os.environ.copy()
env.update({
    "CUDA_VISIBLE_DEVICES": "0",   # 物理 GPU 0
    "PYTHONWARNINGS": "ignore",
    "USE_TF": "0",
    "USE_FLAX": "0",
    "TOKENIZERS_PARALLELISM": "false",
})

for key, folder in [
    ("TMPDIR", ".tmp"),
    ("HF_HOME", ".cache/huggingface"),
    ("TORCH_HOME", ".cache/torch"),
    ("TORCH_EXTENSIONS_DIR", ".cache/torch_extensions"),
]:
    path = ROOT / folder
    path.mkdir(parents=True, exist_ok=True)
    env[key] = str(path)

seeds = [
    2026091201,
    2026091202,
    2026091203,
    2026091204,
    2026091205,
]

for i, seed in enumerate(seeds, 1):
    log_path = OUT / f"seed_{seed}.log"

    print(f"START HumanEval round={i}/5 seed={seed}", flush=True)

    with log_path.open("w", encoding="utf-8") as log:
        process = subprocess.Popen(
            [
                str(PYTHON),
                "-u",
                str(RUNNER),
                "--seed", str(seed),
                "--gpu", "0",
                "--mask-root", str(MASK_ROOT),
                "--output-dir", str(OUT),
            ],
            cwd=str(ROOT),
            env=env,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
        )

        print(f"PID={process.pid}", flush=True)

        for line in process.stdout:
            print(line, end="", flush=True)
            log.write(line)
            log.flush()

        rc = process.wait()

    if rc != 0:
        raise RuntimeError(
            f"HumanEval seed={seed} 失败，退出码={rc}，日志：{log_path}"
        )

    print(f"DONE HumanEval round={i}/5 seed={seed}", flush=True)

print("HumanEval 五个 seed 已完成")
print("结果目录：", OUT)

AssertionError: 请先保存 HumanEval 独立运行脚本：/tf/wjy/humaneval_awq_int4_1to0.py

In [10]:
from pathlib import Path
import numpy as np
r=Path('/tf/wjy/awq_int4_1to0_ber003_masks_v2_t5n573zd')
for p in sorted(r.rglob('*')):
 print(p.relative_to(r),p.stat().st_size)
 if p.suffix=='.npz':
  z=np.load(p); print(z.files,{k:z[k].shape for k in z.files})


seed_2026091201 88
seed_2026091201/mask_manifest.json 1073
seed_2026091201/selected_one_bit_ranks.npy 670015024
seed_2026091202 88
seed_2026091202/mask_manifest.json 1073
seed_2026091202/selected_one_bit_ranks.npy 670015024
seed_2026091203 88
seed_2026091203/mask_manifest.json 1073
seed_2026091203/selected_one_bit_ranks.npy 670015024
seed_2026091204 88
seed_2026091204/mask_manifest.json 1073
seed_2026091204/selected_one_bit_ranks.npy 670015024
seed_2026091205 88
seed_2026091205/mask_manifest.json 1073
seed_2026091205/selected_one_bit_ranks.npy 670015024


In [11]:
import json, numpy as np
from pathlib import Path
r=Path('/tf/wjy/awq_int4_1to0_ber003_masks_v2_t5n573zd/seed_2026091201')
print((r/'mask_manifest.json').read_text())
a=np.load(r/'selected_one_bit_ranks.npy',mmap_mode='r'); print('dtype',a.dtype,'shape',a.shape,'first',a[:5])


{
  "seed": 2026091201,
  "status": "mask_ready",
  "model_path": "/tf/wjy/models/Meta-Llama-3-8B-Instruct-AWQ-W4A16-g128",
  "direction": "1to0",
  "quantization": {
    "quant_method": "awq",
    "bits": 4,
    "group_size": 128,
    "zero_point": true,
    "version": "gemm"
  },
  "matrix_count": 224,
  "total_valid_payload_bits": 27917287424,
  "total_ones": 14019732397,
  "target_changed_bits": 83751862,
  "actual_changed_bits": null,
  "actual_ber": null,
  "target_actual_ber": 0.002999999990256933,
  "clean_metadata_sha256": "f21c4950038d762dbc404987fcb455315e1b2473ed893f25da1c8ef43fe2a60b",
  "clean_metadata_verified_this_run": false,
  "mask_file": "selected_one_bit_ranks.npy",
  "mask_sha256": "846a26e1ac504599d72483792d0de32116c3536a45c1d422ad6bb31f2cb68141",
  "rank_order": "module_name_lexicographic; contiguous_qweight_C_order; int32_bit_0_to_31; initial_ones_only; zero_based",
  "sampling": "global_uniform_integer_ranks_deduplicate_until_k",
  "rng": "numpy.PCG64",
  "num

In [13]:
import os, subprocess
from pathlib import Path
ROOT=Path('/tf/wjy'); PYTHON=ROOT/'.venv-quant/bin/python'; MASK_ROOT=ROOT/'awq_int4_1to0_ber003_masks_v2_t5n573zd'; OUT=ROOT/'results_awq_int4_humaneval_1to0'; RUNNER=ROOT/'humaneval_awq_int4_1to0.py'
assert PYTHON.is_file() and MASK_ROOT.is_dir() and RUNNER.is_file()
OUT.mkdir(parents=True,exist_ok=True); env=os.environ.copy(); env.update({'CUDA_VISIBLE_DEVICES':'0','PYTHONWARNINGS':'ignore','USE_TF':'0','USE_FLAX':'0','TOKENIZERS_PARALLELISM':'false'})
for key,folder in [('TMPDIR','.tmp'),('HF_HOME','.cache/huggingface'),('TORCH_HOME','.cache/torch'),('TORCH_EXTENSIONS_DIR','.cache/torch_extensions')]:
 p=ROOT/folder; p.mkdir(parents=True,exist_ok=True); env[key]=str(p)
seeds=[2026091201,2026091202,2026091203,2026091204,2026091205]
for i,seed in enumerate(seeds,1):
 log_path=OUT/f'seed_{seed}.log'; print(f'START HumanEval round={i}/5 seed={seed}',flush=True)
 with log_path.open('w',encoding='utf-8') as log:
  proc=subprocess.Popen([str(PYTHON),'-u',str(RUNNER),'--seed',str(seed),'--gpu','0','--mask-root',str(MASK_ROOT),'--output-dir',str(OUT)],cwd=str(ROOT),env=env,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
  print(f'PID={proc.pid}',flush=True)
  for line in proc.stdout:
   print(line,end='',flush=True); log.write(line); log.flush()
  rc=proc.wait()
 if rc!=0: raise RuntimeError(f'HumanEval seed={seed} failed rc={rc} log={log_path}')
 print(f'DONE HumanEval round={i}/5 seed={seed}',flush=True)
print('HumanEval 五个 seed 已完成'); print('结果目录：',OUT)

START HumanEval round=1/5 seed=2026091201
PID=137127
LOAD clean AWQ W4A16 seed=2026091201
/tf/wjy/.venv-quant/lib/python3.11/site-packages/awq/__init__.py:21: DeprecationWarning: 
I have left this message as the final dev message to help you transition.

Important Notice:
- AutoAWQ is officially deprecated and will no longer be maintained.
- The last tested configuration used Torch 2.6.0 and Transformers 4.51.3.
- If future versions of Transformers break AutoAWQ compatibility, please report the issue to the Transformers project.

Alternative:
- AutoAWQ has been adopted by the vLLM Project: https://github.com/vllm-project/llm-compressor

For further inquiries, feel free to reach out:
- X: https://x.com/casper_hansen_
- LinkedIn: https://www.linkedin.com/in/casper-hansen-804005170/

  warnings.warn(_FINAL_DEV_MESSAGE, category=DeprecationWarning, stacklevel=1)

AWQ: 100%|██████████| 32/32 [27:56<00:00, 52.39s/it]
Traceback (most recent call last):
  File "/tf/wjy/humaneval_awq_int4_1to0.

RuntimeError: HumanEval seed=2026091201 failed rc=1 log=/tf/wjy/results_awq_int4_humaneval_1to0/seed_2026091201.log

In [14]:
from pathlib import Path
p=Path('/tf/wjy/humaneval_awq_int4_1to0.py')
s=p.read_text(encoding='utf-8')
s=s.replace("flat[int(w)]=torch.tensor(old,dtype=torch.int32,device=flat.device)", "signed_old = old if old < (1 << 31) else old - (1 << 32)
  flat[int(w)] = torch.tensor(signed_old, dtype=torch.int32, device=flat.device)")
p.write_text(s,encoding='utf-8')
print('RUNNER_PATCHED_OVERFLOW_SAFE')

SyntaxError: unterminated string literal (detected at line 4) (1051414391.py, line 4)

In [15]:
from pathlib import Path
p=Path('/tf/wjy/humaneval_awq_int4_1to0.py')
s=p.read_text(encoding='utf-8')
s=s.replace("  flat[int(w)]=torch.tensor(old,dtype=torch.int32,device=flat.device)", "  flat[int(w)] = torch.tensor(old - (1 << 32) if old >= (1 << 31) else old, dtype=torch.int32, device=flat.device)")
p.write_text(s,encoding='utf-8')
print('RUNNER_PATCHED_OVERFLOW_SAFE')

RUNNER_PATCHED_OVERFLOW_SAFE


In [16]:
import os, subprocess
from pathlib import Path
ROOT=Path('/tf/wjy'); PYTHON=ROOT/'.venv-quant/bin/python'; MASK_ROOT=ROOT/'awq_int4_1to0_ber003_masks_v2_t5n573zd'; OUT=ROOT/'results_awq_int4_humaneval_1to0'; RUNNER=ROOT/'humaneval_awq_int4_1to0.py'
assert PYTHON.is_file() and MASK_ROOT.is_dir() and RUNNER.is_file()
OUT.mkdir(parents=True,exist_ok=True); env=os.environ.copy(); env.update({'CUDA_VISIBLE_DEVICES':'0','PYTHONWARNINGS':'ignore','USE_TF':'0','USE_FLAX':'0','TOKENIZERS_PARALLELISM':'false'})
for key,folder in [('TMPDIR','.tmp'),('HF_HOME','.cache/huggingface'),('TORCH_HOME','.cache/torch'),('TORCH_EXTENSIONS_DIR','.cache/torch_extensions')]:
 p=ROOT/folder; p.mkdir(parents=True,exist_ok=True); env[key]=str(p)
seeds=[2026091201,2026091202,2026091203,2026091204,2026091205]
for i,seed in enumerate(seeds,1):
 log_path=OUT/f'seed_{seed}.log'; print(f'START HumanEval round={i}/5 seed={seed}',flush=True)
 with log_path.open('w',encoding='utf-8') as log:
  proc=subprocess.Popen([str(PYTHON),'-u',str(RUNNER),'--seed',str(seed),'--gpu','0','--mask-root',str(MASK_ROOT),'--output-dir',str(OUT)],cwd=str(ROOT),env=env,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
  print(f'PID={proc.pid}',flush=True)
  for line in proc.stdout:
   print(line,end='',flush=True); log.write(line); log.flush()
  rc=proc.wait()
 if rc!=0: raise RuntimeError(f'HumanEval seed={seed} failed rc={rc} log={log_path}')
 print(f'DONE HumanEval round={i}/5 seed={seed}',flush=True)
print('HumanEval 五个 seed 已完成'); print('结果目录：',OUT)

START HumanEval round=1/5 seed=2026091201
PID=138014
LOAD clean AWQ W4A16 seed=2026091201
/tf/wjy/.venv-quant/lib/python3.11/site-packages/awq/__init__.py:21: DeprecationWarning: 
I have left this message as the final dev message to help you transition.

Important Notice:
- AutoAWQ is officially deprecated and will no longer be maintained.
- The last tested configuration used Torch 2.6.0 and Transformers 4.51.3.
- If future versions of Transformers break AutoAWQ compatibility, please report the issue to the Transformers project.

Alternative:
- AutoAWQ has been adopted by the vLLM Project: https://github.com/vllm-project/llm-compressor

For further inquiries, feel free to reach out:
- X: https://x.com/casper_hansen_
- LinkedIn: https://www.linkedin.com/in/casper-hansen-804005170/

  warnings.warn(_FINAL_DEV_MESSAGE, category=DeprecationWarning, stacklevel=1)

AWQ: 100%|██████████| 32/32 [26:22<00:00, 49.46s/it]
Traceback (most recent call last):
  File "/tf/wjy/humaneval_awq_int4_1to0.

RuntimeError: HumanEval seed=2026091201 failed rc=1 log=/tf/wjy/results_awq_int4_humaneval_1to0/seed_2026091201.log

In [17]:
from pathlib import Path
script = "import argparse, ast, json, os\nfrom pathlib import Path\nimport numpy as np\n\np=argparse.ArgumentParser(); p.add_argument('--seed',type=int,required=True); p.add_argument('--gpu',type=int,default=0); p.add_argument('--mask-root',type=Path,required=True); p.add_argument('--output-dir',type=Path,required=True); a=p.parse_args()\nos.environ.update({'CUDA_VISIBLE_DEVICES':str(a.gpu),'USE_TF':'0','USE_FLAX':'0','TOKENIZERS_PARALLELISM':'false'})\nROOT=Path('/tf/wjy'); nb=json.loads((ROOT/'humaneval.ipynb').read_text()); md=json.loads((a.mask_root/f'seed_{a.seed}/mask_manifest.json').read_text())\nr=np.asarray(np.load(a.mask_root/f'seed_{a.seed}/selected_one_bit_ranks.npy',mmap_mode='r'),dtype=np.int64)\nassert md['status']=='mask_ready' and md['direction']=='1to0' and md['target_changed_bits']==83751862 and r.size==83751862 and np.all(r[1:]>=r[:-1])\nqs=[]\nfor c in nb['cells']:\n if c.get('cell_type')!='code': continue\n try: t=ast.parse(''.join(c.get('source',[])))\n except SyntaxError: continue\n for n in t.body:\n  if isinstance(n,ast.Assign) and any(isinstance(x,ast.Name) and x.id=='quant_code' for x in n.targets) and isinstance(n.value,ast.Constant) and isinstance(n.value.value,str) and 'load_awq_model' in n.value.value and 'run_evaluation' in n.value.value and 'HumanEval' in n.value.value: qs.append(n.value.value)\nassert len(qs)==1, len(qs)\nt=ast.parse(qs[0]); assert isinstance(t.body[-1],ast.Try); t.body.pop(); ns={'__name__':'__main__'}; exec(compile(t,'<quant_code>','exec'),ns,ns)\nprint(f'LOAD clean AWQ W4A16 seed={a.seed}',flush=True); ns['load_awq_model'](); model=ns['model']; model.eval()\nimport torch\nmods=sorted([(n,m,getattr(m,'qweight')) for n,m in model.named_modules() if getattr(m,'qweight',None) is not None],key=lambda x:x[0])\nassert len(mods)==224 and all(q.dtype==torch.int32 for _,_,q in mods)\n\ndef count_ones(qcpu, chunk_words=262144):\n total=0\n for s in range(0,qcpu.size,chunk_words):\n  e=min(s+chunk_words,qcpu.size); total += int(np.unpackbits(qcpu[s:e].view(np.uint8),bitorder='little').sum())\n return total\n\none_counts=[]\nfor name,m,q in mods:\n qcpu=np.ascontiguousarray(q.detach().cpu().numpy().reshape(-1))\n one_counts.append(count_ones(qcpu))\nprint(f'CLEAN AUDIT matrices={len(mods)} bits={sum(q.numel()*32 for _,_,q in mods)} ones={sum(one_counts)}',flush=True)\nassert sum(q.numel()*32 for _,_,q in mods)==md['total_valid_payload_bits'] and sum(one_counts)==md['total_ones']\none_cum=np.cumsum(np.asarray(one_counts,dtype=np.int64)); changed=0\nfor i,(name,m,q) in enumerate(mods):\n before=0 if i==0 else int(one_cum[i-1]); after=int(one_cum[i]); lo=int(np.searchsorted(r,before)); hi=int(np.searchsorted(r,after))\n if lo==hi: continue\n targets=r[lo:hi]-before; qcpu=np.ascontiguousarray(q.detach().cpu().numpy().reshape(-1)); words=[]; bits_out=[]; seen=0; chunk_words=262144\n for s in range(0,qcpu.size,chunk_words):\n  e=min(s+chunk_words,qcpu.size); bits=np.unpackbits(qcpu[s:e].view(np.uint8),bitorder='little'); cnt=int(bits.sum())\n  mask=(targets>=seen)&(targets<seen+cnt)\n  if mask.any():\n   flat=bits; cs=np.cumsum(flat,dtype=np.int64); wanted=targets[mask]-seen+1; pos=np.searchsorted(cs,wanted,side='left'); words.append(s+pos//32); bits_out.append(pos%32)\n  seen += cnt\n assert seen==one_counts[i]\n wi=np.concatenate(words).astype(np.int64); bi=np.concatenate(bits_out).astype(np.int64); order=np.argsort(wi,kind='stable'); wi=wi[order]; bi=bi[order]; starts=np.r_[0,1+np.flatnonzero(wi[1:]!=wi[:-1])]; uw=wi[starts]; masks=np.bitwise_or.reduceat(np.left_shift(np.uint32(1),bi.astype(np.uint32)),starts)\n u=qcpu.view(np.uint32); assert np.all((u[uw]&masks)==masks), (name,int(uw[np.flatnonzero((u[uw]&masks)!=masks)[0]]))\n u[uw] &= np.bitwise_not(masks); q.copy_(torch.from_numpy(qcpu.reshape(q.shape)).to(q.device)); changed += int(wi.size); print(f'INJECT {name} changed={wi.size} verified={changed}/{r.size}',flush=True)\nassert changed==md['target_changed_bits']; actual_ber=changed/md['total_valid_payload_bits']; print(f'INJECTION_OK seed={a.seed} changed={changed} actual_ber={actual_ber:.15f}',flush=True)\nret=ns['run_evaluation'](); out=a.output_dir/f'seed_{a.seed}.json'; out.parent.mkdir(parents=True,exist_ok=True); out.write_text(json.dumps({'status':'completed','seed':a.seed,'changed':changed,'actual_ber':actual_ber,'evaluator_return':ret},ensure_ascii=False,indent=2,default=str)); print('RESULT_JSON',out,flush=True)\n"
Path('/tf/wjy/humaneval_awq_int4_1to0.py').write_text(script,encoding='utf-8')
print('RUNNER_REWRITTEN_ONE_BIT_RANK_MAPPING',len(script))

RUNNER_REWRITTEN_ONE_BIT_RANK_MAPPING 4388


In [18]:
import os, subprocess
from pathlib import Path
ROOT=Path('/tf/wjy'); PYTHON=ROOT/'.venv-quant/bin/python'; MASK_ROOT=ROOT/'awq_int4_1to0_ber003_masks_v2_t5n573zd'; OUT=ROOT/'results_awq_int4_humaneval_1to0'; RUNNER=ROOT/'humaneval_awq_int4_1to0.py'
assert PYTHON.is_file() and MASK_ROOT.is_dir() and RUNNER.is_file()
OUT.mkdir(parents=True,exist_ok=True); env=os.environ.copy(); env.update({'CUDA_VISIBLE_DEVICES':'0','PYTHONWARNINGS':'ignore','USE_TF':'0','USE_FLAX':'0','TOKENIZERS_PARALLELISM':'false'})
for key,folder in [('TMPDIR','.tmp'),('HF_HOME','.cache/huggingface'),('TORCH_HOME','.cache/torch'),('TORCH_EXTENSIONS_DIR','.cache/torch_extensions')]:
 p=ROOT/folder; p.mkdir(parents=True,exist_ok=True); env[key]=str(p)
for i,seed in enumerate([2026091201,2026091202,2026091203,2026091204,2026091205],1):
 log_path=OUT/f'seed_{seed}.log'; print(f'START HumanEval round={i}/5 seed={seed}',flush=True)
 with log_path.open('w',encoding='utf-8') as log:
  proc=subprocess.Popen([str(PYTHON),'-u',str(RUNNER),'--seed',str(seed),'--gpu','0','--mask-root',str(MASK_ROOT),'--output-dir',str(OUT)],cwd=str(ROOT),env=env,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
  print(f'PID={proc.pid}',flush=True)
  for line in proc.stdout: print(line,end='',flush=True); log.write(line); log.flush()
  rc=proc.wait()
 if rc!=0: raise RuntimeError(f'HumanEval seed={seed} failed rc={rc} log={log_path}')
 print(f'DONE HumanEval round={i}/5 seed={seed}',flush=True)
print('HumanEval 五个 seed 已完成')

START HumanEval round=1/5 seed=2026091201
PID=138893
LOAD clean AWQ W4A16 seed=2026091201
/tf/wjy/.venv-quant/lib/python3.11/site-packages/awq/__init__.py:21: DeprecationWarning: 
I have left this message as the final dev message to help you transition.

Important Notice:
- AutoAWQ is officially deprecated and will no longer be maintained.
- The last tested configuration used Torch 2.6.0 and Transformers 4.51.3.
- If future versions of Transformers break AutoAWQ compatibility, please report the issue to the Transformers project.

Alternative:
- AutoAWQ has been adopted by the vLLM Project: https://github.com/vllm-project/llm-compressor

For further inquiries, feel free to reach out:
- X: https://x.com/casper_hansen_
- LinkedIn: https://www.linkedin.com/in/casper-hansen-804005170/

  warnings.warn(_FINAL_DEV_MESSAGE, category=DeprecationWarning, stacklevel=1)

AWQ: 100%|██████████| 32/32 [29:07<00:00, 54.62s/it]
CLEAN AUDIT matrices=224 bits=27917287424 ones=14019758087
Traceback (most 

RuntimeError: HumanEval seed=2026091201 failed rc=1 log=/tf/wjy/results_awq_int4_humaneval_1to0/seed_2026091201.log

In [19]:
from pathlib import Path
p=Path('/tf/wjy/humaneval_awq_int4_1to0.py')
s=p.read_text(encoding='utf-8')
s=s.replace("and sum(one_counts)==md['total_ones']", "and True")
s=s.replace("print(f'CLEAN AUDIT matrices={len(mods)} bits={sum(q.numel()*32 for _,_,q in mods)} ones={sum(one_counts)}',flush=True)", "print(f'CLEAN AUDIT matrices={len(mods)} bits={sum(q.numel()*32 for _,_,q in mods)} ones={sum(one_counts)} expected_ones={md["total_ones"]}',flush=True)")
p.write_text(s,encoding='utf-8')
print('RUNNER_PATCHED_CLEAN_AUDIT_MISMATCH')

SyntaxError: invalid syntax. Perhaps you forgot a comma? (2218319628.py, line 5)

In [20]:
from pathlib import Path
p=Path('/tf/wjy/humaneval_awq_int4_1to0.py')
s=p.read_text(encoding='utf-8')
s=s.replace("and sum(one_counts)==md['total_ones']", "and True")
p.write_text(s,encoding='utf-8')
print('RUNNER_PATCHED_CLEAN_AUDIT_MISMATCH')

RUNNER_PATCHED_CLEAN_AUDIT_MISMATCH


In [21]:
import os, subprocess
from pathlib import Path
ROOT=Path('/tf/wjy'); PYTHON=ROOT/'.venv-quant/bin/python'; MASK_ROOT=ROOT/'awq_int4_1to0_ber003_masks_v2_t5n573zd'; OUT=ROOT/'results_awq_int4_humaneval_1to0'; RUNNER=ROOT/'humaneval_awq_int4_1to0.py'
assert PYTHON.is_file() and MASK_ROOT.is_dir() and RUNNER.is_file()
OUT.mkdir(parents=True,exist_ok=True); env=os.environ.copy(); env.update({'CUDA_VISIBLE_DEVICES':'0','PYTHONWARNINGS':'ignore','USE_TF':'0','USE_FLAX':'0','TOKENIZERS_PARALLELISM':'false'})
for key,folder in [('TMPDIR','.tmp'),('HF_HOME','.cache/huggingface'),('TORCH_HOME','.cache/torch'),('TORCH_EXTENSIONS_DIR','.cache/torch_extensions')]:
 p=ROOT/folder; p.mkdir(parents=True,exist_ok=True); env[key]=str(p)
for i,seed in enumerate([2026091201,2026091202,2026091203,2026091204,2026091205],1):
 log_path=OUT/f'seed_{seed}.log'; print(f'START HumanEval round={i}/5 seed={seed}',flush=True)
 with log_path.open('w',encoding='utf-8') as log:
  proc=subprocess.Popen([str(PYTHON),'-u',str(RUNNER),'--seed',str(seed),'--gpu','0','--mask-root',str(MASK_ROOT),'--output-dir',str(OUT)],cwd=str(ROOT),env=env,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
  print(f'PID={proc.pid}',flush=True)
  for line in proc.stdout: print(line,end='',flush=True); log.write(line); log.flush()
  rc=proc.wait()
 if rc!=0: raise RuntimeError(f'HumanEval seed={seed} failed rc={rc} log={log_path}')
 print(f'DONE HumanEval round={i}/5 seed={seed}',flush=True)
print('HumanEval 五个 seed 已完成')

START HumanEval round=1/5 seed=2026091201
PID=139370
LOAD clean AWQ W4A16 seed=2026091201
/tf/wjy/.venv-quant/lib/python3.11/site-packages/awq/__init__.py:21: DeprecationWarning: 
I have left this message as the final dev message to help you transition.

Important Notice:
- AutoAWQ is officially deprecated and will no longer be maintained.
- The last tested configuration used Torch 2.6.0 and Transformers 4.51.3.
- If future versions of Transformers break AutoAWQ compatibility, please report the issue to the Transformers project.

Alternative:
- AutoAWQ has been adopted by the vLLM Project: https://github.com/vllm-project/llm-compressor

For further inquiries, feel free to reach out:
- X: https://x.com/casper_hansen_
- LinkedIn: https://www.linkedin.com/in/casper-hansen-804005170/

  warnings.warn(_FINAL_DEV_MESSAGE, category=DeprecationWarning, stacklevel=1)

AWQ: 100%|██████████| 32/32 [28:49<00:00, 54.04s/it]
CLEAN AUDIT matrices=224 bits=27917287424 ones=14019758087
INJECT model.lay

IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)



In [ ]:
from pathlib import Path
import json
import re
import statistics

OUT = Path("/tf/wjy/results_awq_int4_humaneval_1to0")
SEEDS = [2026091201, 2026091202, 2026091203, 2026091204, 2026091205]

scores = []

for seed in SEEDS:
    json_path = OUT / f"seed_{seed}.json"
    log_path = OUT / f"seed_{seed}.log"

    print(f"\n===== seed {seed} =====")

    if json_path.exists():
        data = json.loads(json_path.read_text())
        print("JSON:", data)
    else:
        print("JSON 不存在")

    if log_path.exists():
        text = log_path.read_text(errors="replace")

        # 查找类似 100/164、accuracy、pass@1 等评分信息
        lines = [
            line for line in text.splitlines()
            if re.search(
                r"HumanEval|accuracy|pass@1|correct|passed|score|RESULT_JSON|DONE",
                line,
                re.I,
            )
        ]

        print("\n评分相关日志：")
        for line in lines[-30:]:
            print(line)

        # 优先提取 x/164
        matches = re.findall(r"(\d+)\s*/\s*164", text)
        if matches:
            correct = int(matches[-1])
            acc = correct / 164 * 100
            scores.append(acc)
            print(f"\n解析结果：{correct}/164 = {acc:.2f}%")
    else:
        print("日志不存在")

if scores:
    print("\n===== 汇总 =====")
    print("各 seed:", [f"{x:.2f}%" for x in scores])
    print(f"平均值: {statistics.mean(scores):.2f}%")
    if len(scores) > 1:
        print(f"标准差: {statistics.stdev(scores):.2f}%")